# SL/TP berbasis ATR vs Fixed Percentage — Strategi V5

**Temuan**: strategi V5 saat ini pakai SL/TP **fixed percentage** untuk SEMUA coin
(`STOP_LOSS=-2%`, `TAKE_PROFIT=+5%`, lihat `strategy_v5.py`). Tidak ada implementasi ATR
sama sekali di codebase — ini kontradiksi dengan rekomendasi dokumen `Time Series Analysis`
(Pilar 3, volatility clustering) dan `Dasar Matematik Analys Treding` (Pilar 2, position
sizing): SL/TP seharusnya menyesuaikan volatilitas tiap coin, bukan angka sama rata.

**Bukti masalah** (dari audit sebelumnya): DOGEUSDT (std return harian 0.1097, kurtosis 817)
vs ETHUSDT (std 0.0409, kurtosis 5) diberi SL -2% yang SAMA — untuk DOGE itu cuma noise
harian biasa, untuk ETH itu pergerakan yang lebih berarti.

**Tujuan notebook ini**: bangun ATR, hitung SL/TP dinamis (`SL = entry - k×ATR`,
`TP = entry + m×ATR`), lalu bandingkan hasil backtest ATR-based vs fixed percentage pakai
gate BUY V5 yang SAMA PERSIS (bottoming + reversal_confirmed + regime_ok) — supaya
perbandingannya adil, cuma variabel exit yang berubah.

Metodologi sama seperti audit sebelumnya: Z-test + EV, dan kalau ada banyak kombinasi
multiplier dicoba, pakai koreksi Bonferroni (pelajaran dari eksperimen regime filter kemarin
yang bikin kandidat "kelihatan bagus" ternyata gagal lolos koreksi statistik ketat).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))              # -> from src.strategy ...
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))  # -> from data_source ...

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

from data_source.loader import load_symbol, discover_symbols
from data_source.fear_greed import load_fear_greed_index
from src.strategy import v5

pd.set_option("display.width", 120)
print("Setup OK — project root:", PROJECT_ROOT)

Setup OK — project root: D:\Projects\backtes-crypto


## 1. Implementasi ATR (Average True Range)

Rumus standar (Wilder):

```
True Range = max(high-low, |high-prev_close|, |low-prev_close|)
ATR(n)     = rolling mean dari True Range, window n hari (biasanya 14)
```

Belum ada implementasi ATR di codebase (`src/crypto_backtest/indicators/`) — dibangun di sini
dulu untuk eksperimen, baru dipindah ke source kalau terbukti valid.

In [2]:
def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr


def atr(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> pd.Series:
    tr = true_range(high, low, close)
    return tr.rolling(window=window, min_periods=window).mean()


def atr_pct(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> pd.Series:
    """ATR sebagai persentase dari harga -- lebih gampang dibandingkan lintas coin & waktu."""
    return atr(high, low, close, window) / close


# Cek cepat: ATR% BTC vs DOGE (harus konsisten dgn kurtosis/std dari audit sebelumnya)
btc_df = load_symbol("BTCUSDT", "1d", "full")
doge_df = load_symbol("DOGEUSDT", "1d", "full")
btc_atr_pct = atr_pct(btc_df["high"], btc_df["low"], btc_df["close"]).dropna()
doge_atr_pct = atr_pct(doge_df["high"], doge_df["low"], doge_df["close"]).dropna()
print(f"BTC  ATR% rata-rata : {btc_atr_pct.mean():.3%}  (median {btc_atr_pct.median():.3%})")
print(f"DOGE ATR% rata-rata : {doge_atr_pct.mean():.3%}  (median {doge_atr_pct.median():.3%})")
print(f"\n-> DOGE ATR% {doge_atr_pct.mean()/btc_atr_pct.mean():.1f}x lebih besar dari BTC "
      f"(konsisten dgn kurtosis 817 vs 9.25 dari audit sebelumnya)")

BTC  ATR% rata-rata : 4.799%  (median 4.272%)
DOGE ATR% rata-rata : 8.326%  (median 7.101%)

-> DOGE ATR% 1.7x lebih besar dari BTC (konsisten dgn kurtosis 817 vs 9.25 dari audit sebelumnya)


## 2. Load semua coin & bangun gate BUY V5 (sama seperti audit sebelumnya)

In [3]:
from src.strategy.scoring.score import build_feature_frame
from src.strategy.scoring.reversal import (
    compute_reversal_scores, add_reversal_confirmation, add_regime_filter,
)

coins = ["ADAUSDT", "AVAXUSDT", "BNBUSDT", "BTCUSDT", "DOGEUSDT", "ETHUSDT", "HBARUSDT", "SOLUSDT", "SUIUSDT", "XRPUSDT"]
fng = load_fear_greed_index()

all_dfs = {}
for s in coins:
    try:
        all_dfs[s] = load_symbol(s, "1d", "full")
    except Exception:
        pass

feats_full = {}
for s, d in all_dfs.items():
    feat = build_feature_frame(d["close"], d["volume"], btc_df["close"], fng)
    feat = compute_reversal_scores(feat)
    feat = add_reversal_confirmation(feat, d["high"], d["low"], d["close"])
    feat = add_regime_filter(feat, d["close"])
    feat["atr_pct"] = atr_pct(d["high"], d["low"], d["close"])
    feats_full[s] = feat

print(f"Coin: {list(all_dfs.keys())} ({len(all_dfs)} coin)")

entry_counts = {s: int((f["bottoming"] & f["reversal_confirmed"] & f["regime_ok"]).sum())
                for s, f in feats_full.items()}
print(f"Total entry gate-penuh per coin: {entry_counts}")
print(f"Total entry gate-penuh (semua coin): {sum(entry_counts.values())}")

Coin: ['ADAUSDT', 'AVAXUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'HBARUSDT', 'SOLUSDT', 'SUIUSDT', 'XRPUSDT'] (10 coin)
Total entry gate-penuh per coin: {'ADAUSDT': 16, 'AVAXUSDT': 13, 'BNBUSDT': 10, 'BTCUSDT': 41, 'DOGEUSDT': 14, 'ETHUSDT': 11, 'HBARUSDT': 11, 'SOLUSDT': 10, 'SUIUSDT': 5, 'XRPUSDT': 14}
Total entry gate-penuh (semua coin): 145


## 3. Simulasi exit day-by-day (lebih realistis dari clamp forward_return)

Catatan penting: `api.py` saat ini menghitung backtest dengan cara **clamp** forward_return
ke `[STOP_LOSS, TAKE_PROFIT]` di akhir horizon — ini simplifikasi kasar yang TIDAK mengecek
apakah SL/TP sebenarnya tersentuh di tengah jalan (hari ke-1, ke-2, dst). Untuk perbandingan
yang adil antara fixed % vs ATR, di sini disimulasikan cek harga **high/low intraday tiap
hari** selama holding period — siapa yang tersentuh duluan (SL atau TP) itu yang menentukan
exit, baru kalau belum ada yang tersentuh sampai horizon habis, keluar di close hari terakhir.

Ini juga PERBAIKAN metodologi dibanding `api.py` — akan direkomendasikan dipindah ke situ
kalau ATR-based terbukti lebih baik.

In [4]:
def simulate_trades(feat: pd.DataFrame, ohlc: pd.DataFrame, entry_mask: pd.Series,
                     sl_pct_fn, tp_pct_fn, horizon: int = v5.FORWARD_HORIZON):
    """Simulasi day-by-day: SL/TP sebagai % (bisa fixed atau berbasis ATR per entry).

    sl_pct_fn(row) -> float negatif (misal -0.02), tp_pct_fn(row) -> float positif (misal 0.05)
    dihitung PER ENTRY (row hari itu), supaya ATR-based bisa beda tiap kejadian.

    Return DataFrame satu baris per entry: entry_date, exit_day, exit_reason
    (SL/TP/HORIZON), realized_return.
    """
    close = ohlc["close"].reindex(feat.index)
    high = ohlc["high"].reindex(feat.index)
    low = ohlc["low"].reindex(feat.index)
    idx = feat.index
    n = len(idx)
    pos_of = {d: i for i, d in enumerate(idx)}
    entry_dates = feat.index[entry_mask.fillna(False)]

    rows = []
    for entry_date in entry_dates:
        i_entry = pos_of[entry_date]
        i_horizon_end = i_entry + horizon
        if i_horizon_end >= n:
            continue
        row = feat.loc[entry_date]
        price_entry = close.iloc[i_entry]
        if pd.isna(price_entry):
            continue
        sl_pct = sl_pct_fn(row)
        tp_pct = tp_pct_fn(row)
        if pd.isna(sl_pct) or pd.isna(tp_pct):
            continue
        sl_price = price_entry * (1 + sl_pct)
        tp_price = price_entry * (1 + tp_pct)

        exit_day = None
        exit_reason = "HORIZON"
        exit_price = close.iloc[i_horizon_end]
        for k in range(1, horizon + 1):
            i_check = i_entry + k
            if i_check >= n:
                break
            hi = high.iloc[i_check]
            lo = low.iloc[i_check]
            if pd.isna(hi) or pd.isna(lo):
                continue
            hit_sl = lo <= sl_price
            hit_tp = hi >= tp_price
            if hit_sl and hit_tp:
                # Ambigu (SL & TP sama-sama tersentuh dalam 1 hari) -> asumsi konservatif: SL duluan
                exit_day, exit_reason, exit_price = k, "SL", sl_price
                break
            elif hit_sl:
                exit_day, exit_reason, exit_price = k, "SL", sl_price
                break
            elif hit_tp:
                exit_day, exit_reason, exit_price = k, "TP", tp_price
                break
        if exit_day is None:
            exit_day = horizon
            if pd.isna(exit_price):
                continue

        realized_return = (exit_price - price_entry) / price_entry
        rows.append({
            "entry_date": entry_date, "exit_day": exit_day, "exit_reason": exit_reason,
            "sl_pct": sl_pct, "tp_pct": tp_pct, "realized_return": realized_return,
        })

    return pd.DataFrame(rows)


def eval_trades(trades: pd.DataFrame, label: str):
    n = len(trades)
    if n == 0:
        print(f"{label}: n=0")
        return None
    wins = (trades["realized_return"] > 0).sum()
    p_obs = wins / n
    se = np.sqrt(0.5 * 0.5 / n)
    z = (p_obs - 0.5) / se if se > 0 else 0.0
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    ev = trades["realized_return"].mean()
    win_r = trades.loc[trades["realized_return"] > 0, "realized_return"]
    loss_r = trades.loc[trades["realized_return"] <= 0, "realized_return"]
    sharpe = ev / trades["realized_return"].std(ddof=1) if trades["realized_return"].std(ddof=1) > 0 else float("nan")
    reason_counts = trades["exit_reason"].value_counts().to_dict()
    print(f"{label}")
    print(f"  n={n}  win_rate={p_obs:.1%}  Z={z:+.2f}  p={p_val:.4f}  EV(mean return)={ev:+.3%}  "
          f"Sharpe(non-annualized)={sharpe:.3f}")
    print(f"  Exit reason: {reason_counts}")
    return {"n": n, "win_rate": float(p_obs), "z": float(z), "p_value": float(p_val),
            "ev": float(ev), "sharpe": float(sharpe), "exit_reasons": reason_counts}


print("Simulation functions ready.")

Simulation functions ready.


## 4. Baseline: fixed -2%/+5% (cara V5 saat ini), tapi disimulasikan day-by-day (bukan clamp)

In [5]:
all_trades_fixed = []
for s, feat in feats_full.items():
    entry_mask = feat["bottoming"] & feat["reversal_confirmed"] & feat["regime_ok"]
    ohlc = all_dfs[s]
    trades = simulate_trades(
        feat, ohlc, entry_mask,
        sl_pct_fn=lambda row: v5.STOP_LOSS,
        tp_pct_fn=lambda row: v5.TAKE_PROFIT,
    )
    if len(trades):
        trades["symbol"] = s
        all_trades_fixed.append(trades)

trades_fixed = pd.concat(all_trades_fixed, ignore_index=True) if all_trades_fixed else pd.DataFrame()
result_fixed = eval_trades(trades_fixed, "BASELINE: fixed SL=-2% / TP=+5% (day-by-day simulation)")

BASELINE: fixed SL=-2% / TP=+5% (day-by-day simulation)
  n=137  win_rate=22.6%  Z=-6.41  p=0.0000  EV(mean return)=-0.409%  Sharpe(non-annualized)=-0.139
  Exit reason: {'SL': 105, 'TP': 31, 'HORIZON': 1}


## 5. ATR-based: SL = entry - k×ATR, TP = entry + m×ATR

Sweep beberapa kombinasi multiplier `k` (SL) dan `m` (TP), sekaligus jaga rasio risk:reward
mirip baseline (~1:2.5) supaya perbandingan adil dari segi struktur, bukan cuma "TP dilebarkan
sampai menang".

In [6]:
atr_multiplier_pairs = [
    (0.5, 1.25),   # rasio 1:2.5, sama seperti baseline
    (1.0, 2.5),
    (1.5, 3.75),
    (0.75, 1.875),
    (1.0, 1.5),    # rasio lebih konservatif 1:1.5
    (1.5, 2.0),    # rasio 1:1.33 (lebih longgar SL, TP lebih dekat)
]

sweep_rows = []
all_trades_by_combo = {}
for k, m in atr_multiplier_pairs:
    combo_trades = []
    for s, feat in feats_full.items():
        entry_mask = feat["bottoming"] & feat["reversal_confirmed"] & feat["regime_ok"]
        ohlc = all_dfs[s]

        def sl_fn(row, k=k):
            a = row.get("atr_pct")
            return -k * a if pd.notna(a) else float("nan")

        def tp_fn(row, m=m):
            a = row.get("atr_pct")
            return m * a if pd.notna(a) else float("nan")

        trades = simulate_trades(feat, ohlc, entry_mask, sl_fn, tp_fn)
        if len(trades):
            trades["symbol"] = s
            combo_trades.append(trades)
    combined = pd.concat(combo_trades, ignore_index=True) if combo_trades else pd.DataFrame()
    all_trades_by_combo[(k, m)] = combined
    res = eval_trades(combined, f"ATR-based: SL=-{k}xATR / TP=+{m}xATR")
    if res:
        res["k"] = k
        res["m"] = m
        sweep_rows.append(res)
    print()

sweep_atr_df = pd.DataFrame(sweep_rows)

ATR-based: SL=-0.5xATR / TP=+1.25xATR
  n=137  win_rate=27.7%  Z=-5.21  p=0.0000  EV(mean return)=-0.231%  Sharpe(non-annualized)=-0.037
  Exit reason: {'SL': 94, 'TP': 31, 'HORIZON': 12}

ATR-based: SL=-1.0xATR / TP=+2.5xATR
  n=137  win_rate=44.5%  Z=-1.28  p=0.2000  EV(mean return)=+0.713%  Sharpe(non-annualized)=0.082
  Exit reason: {'HORIZON': 68, 'SL': 55, 'TP': 14}

ATR-based: SL=-1.5xATR / TP=+3.75xATR
  n=137  win_rate=53.3%  Z=+0.77  p=0.4419  EV(mean return)=+1.342%  Sharpe(non-annualized)=0.128
  Exit reason: {'HORIZON': 88, 'SL': 41, 'TP': 8}

ATR-based: SL=-0.75xATR / TP=+1.875xATR
  n=137  win_rate=36.5%  Z=-3.16  p=0.0016  EV(mean return)=+0.030%  Sharpe(non-annualized)=0.004
  Exit reason: {'SL': 74, 'HORIZON': 44, 'TP': 19}

ATR-based: SL=-1.0xATR / TP=+1.5xATR
  n=137  win_rate=46.7%  Z=-0.77  p=0.4419  EV(mean return)=+0.635%  Sharpe(non-annualized)=0.081
  Exit reason: {'HORIZON': 54, 'SL': 53, 'TP': 30}



ATR-based: SL=-1.5xATR / TP=+2.0xATR
  n=137  win_rate=54.7%  Z=+1.11  p=0.2667  EV(mean return)=+1.140%  Sharpe(non-annualized)=0.124
  Exit reason: {'HORIZON': 75, 'SL': 40, 'TP': 22}



## 6. Rangkuman & koreksi Bonferroni

6 kombinasi ATR dicoba sekaligus dengan baseline fixed — pakai koreksi Bonferroni seperti
eksperimen regime filter sebelumnya, supaya kandidat terbaik tidak cuma "kelihatan bagus"
karena kebetulan mencoba banyak kombinasi (data snooping, Pilar 4 PDF).

In [7]:
n_combos = len(atr_multiplier_pairs) + 1  # +1 untuk baseline fixed
bonferroni_alpha = 0.05 / n_combos
bonferroni_z = stats.norm.ppf(1 - bonferroni_alpha / 2)
print(f"Total kombinasi dicoba (termasuk baseline): {n_combos}  ->  "
      f"Bonferroni Z threshold: {bonferroni_z:.2f} (vs standar 1.96)\n")

summary_rows = [{"label": "BASELINE fixed -2%/+5%", "k": None, "m": None, **result_fixed}]
for _, r in sweep_atr_df.iterrows():
    summary_rows.append({"label": f"ATR SL=-{r['k']}x / TP=+{r['m']}x", **r.to_dict()})

summary_df = pd.DataFrame(summary_rows)[["label", "n", "win_rate", "z", "p_value", "ev", "sharpe"]]
summary_df["significant_standard"] = summary_df["z"].abs() > 1.96
summary_df["significant_bonferroni"] = summary_df["z"].abs() > bonferroni_z
summary_df["valid_standard"] = summary_df["significant_standard"] & (summary_df["ev"] > 0)
summary_df["valid_bonferroni"] = summary_df["significant_bonferroni"] & (summary_df["ev"] > 0)

print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print(f"\nKombinasi valid (standar Z>1.96, EV+): {summary_df['valid_standard'].sum()} dari {len(summary_df)}")
print(f"Kombinasi valid (Bonferroni Z>{bonferroni_z:.2f}, EV+): {summary_df['valid_bonferroni'].sum()} dari {len(summary_df)}")

Total kombinasi dicoba (termasuk baseline): 7  ->  Bonferroni Z threshold: 2.69 (vs standar 1.96)

                     label   n  win_rate       z  p_value      ev  sharpe  significant_standard  significant_bonferroni  valid_standard  valid_bonferroni
    BASELINE fixed -2%/+5% 137    0.2263 -6.4077   0.0000 -0.0041 -0.1392                  True                    True           False             False
  ATR SL=-0.5x / TP=+1.25x 137    0.2774 -5.2116   0.0000 -0.0023 -0.0374                  True                    True           False             False
   ATR SL=-1.0x / TP=+2.5x 137    0.4453 -1.2815   0.2000  0.0071  0.0817                 False                   False           False             False
  ATR SL=-1.5x / TP=+3.75x 137    0.5328  0.7689   0.4419  0.0134  0.1277                 False                   False           False             False
ATR SL=-0.75x / TP=+1.875x 137    0.3650 -3.1611   0.0016  0.0003  0.0041                  True                    True            

## 7. Kesimpulan

**Temuan #1 (paling kritis)**: baseline fixed -2%/+5% yang dipakai PRODUKSI SAAT INI, ketika
disimulasikan day-by-day secara realistis (bukan clamp forward_return), terbukti **SIGNIFIKAN
NEGATIF** (Z=-6.41, EV=-0.41% per trade, win rate cuma 22.6%). SL -2% terlalu ketat untuk
mayoritas coin — 105 dari 137 trade (77%) kena stop-loss sebelum sempat reversal, karena -2%
itu berada DALAM rentang noise harian normal untuk coin crypto (ATR% rata-rata BTC saja 4.8%,
jauh lebih besar dari 2%). Ini dibahas lebih detail di bagian 8.

**Temuan #2 (soal ATR)**: dari 6 kombinasi ATR yang dicoba + baseline, cuma 1 yang "valid"
secara formal (signifikan + EV positif), tapi EV-nya nyaris nol (+0.03%) — bukan kandidat yang
menarik. Kombinasi yang EV-nya paling besar dan menjanjikan (SL=-1.5×ATR/TP=+3.75×ATR, EV
+1.34%; SL=-1.5×ATR/TP=+2.0×ATR, EV +1.14%) **TIDAK signifikan secara statistik** (p=0.44,
p=0.27) — sample n=137 terlalu kecil untuk memastikan ini bukan kebetulan.

**Kesimpulan jujur**: ATR-based SL/TP dengan multiplier lebih longgar (k≈1.5, m≈2-3.75)
**terlihat jauh lebih baik** daripada fixed -2%/+5% (mengubah dari EV signifikan negatif jadi
EV positif meski belum signifikan) — tapi belum bisa diklaim "terbukti valid", cuma
"menjanjikan, perlu lebih banyak data untuk konfirmasi". Yang sudah pasti secara statistik:
**baseline fixed saat ini bermasalah** dan sebaiknya tidak dipertahankan apa adanya.

**Rekomendasi**: prioritas pertama adalah memperbaiki metodologi backtest (lihat bagian 8) —
supaya visibility performa V5 yang sesungguhnya diketahui dengan benar, baru setelah itu
memutuskan parameter SL/TP pengganti (fixed yang dilonggarkan, atau ATR-based) berdasarkan
data yang bertambah banyak seiring waktu.

---

## 8. Investigasi bug: kenapa `api.py` klaim win rate tinggi, padahal simulasi realistis negatif?

Bandingkan LANGSUNG dua metode pada entry & data yang SAMA PERSIS:

- **Metode lama** (`api.py` baris ~392): `realized = clamp(forward_return, STOP_LOSS, TAKE_PROFIT)`
  — cuma melihat harga di HARI KE-5 (akhir horizon), lalu dipotong ke rentang [-2%, +5%].
  Tidak pernah mengecek apakah harga sempat turun >2% di hari ke-1/2/3/4 lalu **reversal
  balik naik** sebelum hari ke-5. Kalau itu terjadi, metode lama tetap mencatat untung
  (karena forward_return di hari-5 positif), padahal di real trading posisi SUDAH ke-stop out
  duluan di harga rugi -2%.
- **Metode baru** (day-by-day, bagian 3 di atas): cek high/low intraday tiap hari, exit di
  hari SL/TP pertama kali tersentuh — ini yang terjadi di real trading.

In [8]:
from src.strategy.backtest_engine import add_forward_return_target

# Replikasi PERSIS cara api.py menghitung "realized_return" (baris 391-392 di api.py)
old_method_rows = []
for s, feat in feats_full.items():
    entry_mask = feat["bottoming"] & feat["reversal_confirmed"] & feat["regime_ok"]
    close = all_dfs[s]["close"]
    labeled = add_forward_return_target(feat, close, horizon=v5.FORWARD_HORIZON)
    sel = labeled[entry_mask.fillna(False) & labeled["forward_return"].notna()]
    for date, row in sel.iterrows():
        fwd = float(row["forward_return"])
        realized_old = max(v5.STOP_LOSS, min(v5.TAKE_PROFIT, fwd))  # PERSIS baris 392 api.py
        old_method_rows.append({"entry_date": date, "symbol": s,
                                 "forward_return_raw": fwd, "realized_old_method": realized_old})

old_method_df = pd.DataFrame(old_method_rows)
print(f"Metode LAMA (clamp forward_return, cara api.py):")
n = len(old_method_df)
wins_old = (old_method_df["realized_old_method"] > 0).sum()
print(f"  n={n}  win_rate={wins_old/n:.1%}  EV(mean)={old_method_df['realized_old_method'].mean():+.3%}")

print(f"\nMetode BARU (day-by-day, bagian 4 di atas):")
print(f"  n={result_fixed['n']}  win_rate={result_fixed['win_rate']:.1%}  EV(mean)={result_fixed['ev']:+.3%}")

# Gabungkan buat lihat kasus spesifik yang beda hasil
merged = trades_fixed.merge(old_method_df, on=["entry_date", "symbol"], how="inner")
merged["old_says_win"] = merged["realized_old_method"] > 0
merged["new_says_win"] = merged["realized_return"] > 0
disagree = merged[merged["old_says_win"] != merged["new_says_win"]]
print(f"\nJumlah trade yang HASILNYA BEDA antara 2 metode: {len(disagree)} dari {len(merged)}")
print(f"\nContoh kasus (metode lama bilang UNTUNG, tapi day-by-day sebenarnya kena SL duluan):")
example = disagree[(disagree["old_says_win"]) & (~disagree["new_says_win"])].head(5)
print(example[["entry_date", "symbol", "exit_day", "exit_reason",
               "realized_return", "forward_return_raw", "realized_old_method"]].to_string(index=False))

Metode LAMA (clamp forward_return, cara api.py):
  n=137  win_rate=58.4%  EV(mean)=+1.407%

Metode BARU (day-by-day, bagian 4 di atas):
  n=137  win_rate=22.6%  EV(mean)=-0.409%

Jumlah trade yang HASILNYA BEDA antara 2 metode: 57 dari 137

Contoh kasus (metode lama bilang UNTUNG, tapi day-by-day sebenarnya kena SL duluan):
entry_date  symbol  exit_day exit_reason  realized_return  forward_return_raw  realized_old_method
2021-07-17 ADAUSDT         2          SL            -0.02            0.011518             0.011518
2021-07-18 ADAUSDT         1          SL            -0.02            0.022348             0.022348
2021-11-27 ADAUSDT         1          SL            -0.02            0.114064             0.050000
2025-02-25 ADAUSDT         1          SL            -0.02            0.661300             0.050000
2025-04-07 ADAUSDT         1          SL            -0.02            0.124659             0.050000


## 9. Kesimpulan bug — SANGAT KRITIS, perlu tindakan segera

**Bukti telak**: metode lama (`api.py`, clamp) mencatat win rate **58.4%**, EV **+1.41%**.
Metode realistis (day-by-day, cek high/low intraday) pada DATA & ENTRY YANG SAMA PERSIS
mencatat win rate **22.6%**, EV **-0.41%**. Selisih **35.8 poin persentase win rate** — dan
**57 dari 137 trade (42%) hasilnya benar-benar bertolak belakang** antara dua metode.

**Akar masalah**: `realized = max(STOP_LOSS, min(TAKE_PROFIT, forward_return))` di
`api.py` baris 392 cuma melihat harga di **hari ke-5 (akhir horizon)**, TIDAK PERNAH mengecek
apakah harga sempat menyentuh level stop-loss di hari 1/2/3/4 lebih dulu. Contoh nyata di atas
(ADAUSDT, entry 2025-02-25): hari pertama harga sudah turun cukup untuk stop-out di -2%, tapi
karena di HARI KE-5 harga sudah rebound +66%, metode lama malah mencatatnya sebagai untung
+5% (clamped). Di real trading dengan stop-loss order sungguhan di exchange, posisi itu SUDAH
keluar rugi -2% jauh sebelum sempat melihat rebound tersebut.

**Implikasi serius**: klaim di komentar `strategy_v5.py` ("Backtest 2024-2026: 73.7% win rate,
+8.97% return") kemungkinan besar dihasilkan dengan metodologi clamp yang sama (perlu
diverifikasi terpisah, karena skrip yang menghasilkan angka itu tidak ikut diaudit di sini) —
**kalau iya, angka itu TIDAK merepresentasikan performa nyata** ketika SL/TP dieksekusi
sebagai order sungguhan di exchange oleh `backend-tredin-rust`.

**Rekomendasi tindakan (prioritas urut)**:
1. **Audit ulang** `train_v5_model.py` dan proses validasi yang menghasilkan klaim "73.7%
   win rate" — cek apakah pakai metode clamp yang sama atau metode lain.
2. **Ganti metodologi backtest** di `api.py` (endpoint `/backtest`) dari clamp ke simulasi
   day-by-day yang realistis (kode `simulate_trades` di notebook ini bisa dipindah ke
   `src/crypto_backtest/backtest/engine.py`).
3. **Setelah metodologi diperbaiki**, baru evaluasi ulang apakah gate BUY V5 (bottoming +
   reversal_confirmed + regime_ok) masih valid dengan SL/TP yang benar-benar direalisasikan
   — dan baru putuskan parameter SL/TP pengganti (ATR-based menjanjikan tapi belum
   signifikan secara statistik, lihat bagian 7).

**INI TIDAK MENGUBAH APAPUN DI PRODUKSI** — semua di atas masih eksperimen/investigasi di
notebook. Perlu keputusan & konfirmasi eksplisit sebelum mengubah `api.py` atau
`strategy_v5.py`.